# Modélisation et Évaluation (Prédiction des Prix Immobiliers)

Ce notebook fait suite à l'analyse exploratoire et au nettoyage des données. L'objectif est d'entraîner plusieurs algorithmes de Machine Learning (régression linéaire et arbres), de comparer leurs performances, et d'optimiser le meilleur modèle pour un futur déploiement.

### 1. Importation et Préparation Stratifiée des Données

**Choix techniques pour la préparation :**
- **Stratification (qcut) :** Pour garantir que le jeu d'entraînement et le jeu de test contiennent la même proportion de biens abordables et de biens luxueux, nous découpons la variable cible (`log_price`) en 5 tranches équilibrées lors de la séparation.
- **Target Encoding :** Le code postal (`zipcode`) est remplacé par le prix moyen du quartier, calculé *strictement* sur le jeu d'entraînement pour éviter la fuite de données.

In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

### 1. Importation et Préparation des Données

In [7]:
# 1. Chargement des données nettoyées
df = pd.read_csv('../data/king_county_cleaned.csv')

# 2. Suppression des colonnes non prédictives
cols_to_drop = ['Unnamed: 0', 'id', 'date', 'price', 'yr_built', 'yr_renovated']
df_model = df.drop(columns=cols_to_drop, errors='ignore')

# 3. Séparation des features (X) et de la cible (y)
X = df_model.drop(columns=['log_price'])
y = df_model['log_price']

# 4. Stratification : création de 5 tranches de prix équilibrées
strat_bins = pd.qcut(y, q=5, labels=False)

# 5. Séparation Train / Test stratifiée (80 % / 20 %)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=strat_bins
)

# 6. Target Encoding sur le 'zipcode'
zipcode_target_mean = y_train.groupby(X_train['zipcode']).mean()
X_train['zipcode_encoded'] = X_train['zipcode'].map(zipcode_target_mean)
X_test['zipcode_encoded'] = X_test['zipcode'].map(zipcode_target_mean)

# Remplacement des éventuelles valeurs inconnues dans le Test
X_test['zipcode_encoded'] = X_test['zipcode_encoded'].fillna(y_train.mean())

# Suppression de la colonne d'origine
X_train = X_train.drop(columns=['zipcode'])
X_test = X_test.drop(columns=['zipcode'])

print(f"X_train stratifié : {X_train.shape}")
print(f"X_test stratifié  : {X_test.shape}")

X_train stratifié : (17126, 20)
X_test stratifié  : (4282, 20)


### 2. Modèle 1 : Régression Linéaire Régularisée (Ridge)

Nous débutons par un modèle linéaire pénalisé (Ridge).

**Explication des choix :**
- **RobustScaler :** Les algorithmes linéaires sont très sensibles à l'échelle. `RobustScaler` utilise la médiane et l'écart interquartile, offrant une mise à l'échelle robuste face aux propriétés exceptionnelles (asymétrie) qui subsistent sur le marché.
- **Validation Croisée In-Memory :** Notre jeu de données tient entièrement en mémoire vive (Full-Batch). `RidgeCV` découpe automatiquement les données d'entraînement en 5 plis croisés pour trouver la meilleure pénalité $\alpha$, sans nécessiter d'apprentissage par petits lots.
- **Métriques (Régression) :** Nous utilisons le **R²** (variance expliquée) et la **MAE** (erreur moyenne absolue reconvertie en dollars), qui sont les standards métiers, plutôt que des métriques de classification inadaptées ici.

In [8]:
# 1. Grille de pénalité alpha
alphas = np.logspace(-3, 3, 100)

# 2. Pipeline Ridge avec RobustScaler
ridge_pipeline = Pipeline([
    ('scaler', RobustScaler()),  
    ('model', RidgeCV(alphas=alphas, cv=5, scoring='neg_mean_squared_error'))
])

# 3. Entraînement sur le jeu Train
ridge_pipeline.fit(X_train, y_train)

best_alpha = ridge_pipeline.named_steps['model'].alpha_
print(f"Meilleur paramètre alpha retenu par RidgeCV : {best_alpha:.4f}\n")

# 4. Prédictions sur le Test et reconversion
y_pred_log = ridge_pipeline.predict(X_test)
y_test_real = np.exp(y_test)
y_pred_real = np.exp(y_pred_log)

# 5. Évaluation
r2_log = r2_score(y_test, y_pred_log)
mae_dollars = mean_absolute_error(y_test_real, y_pred_real)
rmse_dollars = np.sqrt(mean_squared_error(y_test_real, y_pred_real))

print("--- PERFORMANCE RIDGE (RobustScaler) ---")
print(f"Score R² : {r2_log:.4f}")
print(f"Erreur Moyenne Absolue (MAE) : {mae_dollars:,.2f} $")
print(f"Erreur Quadratique (RMSE)    : {rmse_dollars:,.2f} $")

Meilleur paramètre alpha retenu par RidgeCV : 1.8738

--- PERFORMANCE RIDGE (RobustScaler) ---
Score R² : 0.8607
Erreur Moyenne Absolue (MAE) : 83,428.44 $
Erreur Quadratique (RMSE)    : 163,671.49 $
